# Análisis de journeys de Usuario
---

<div style="text-align: center">
    <img src="https://raw.githubusercontent.com/ljpiere/tpdata_python/main/images/w1s1_2.png" width="400">
</div>

## Agenda y Objetivos de Aprendizaje


1. Comprender los *user journey* 
2. Realizar un análisis de embudo usando SQL
3. Analizar retención con cohortes

## *User Journey*

---

Un *user journey* es la representación de un proceso multi-etapa visto desde la perspectiva del usuario (ej.: descubrimiento → evaluación → conversión → uso/retención).  
Sirve para conectar métricas de producto con impacto de negocio (tasa de conversión, *drop-off*, tiempo entre etapas, LTV).

**Ejercicio (discursivo):** Analiza 3 journeys y define etapas + eventos:
- **E-commerce:** visita → *view_item* → *add_to_cart* → *begin_checkout* → *purchase*  
- **Educación:** *landing* → registro → primera lección → completar módulo → certificación  
- **Servicios públicos:** ingresar a portal → solicitar turno → adjuntar documentos → pago → confirmación


### Identificar eventos y campos clave en BD
Campos mínimos para reconstruir journeys:  
- `user_id` (agrupar por usuario)  
- `event_ts` (orden temporal)  
- `event_name` (tipo de evento)  
- `props` (JSONB con detalles: sku, valor, utm, etc.)

**Ejemplo: listar tipos de evento únicos**
```sql
SELECT DISTINCT event_name
FROM events
ORDER BY 1;
```
**Ejercicio:** Revisa la tabla `events` y marca cuáles campos sirven para ordenar eventos, agrupar usuarios y calcular conversiones.


###  Validación de calidad de datos
Antes de construir consultas:
- **Duplicados:** ¿múltiples eventos idénticos?  
- **Faltantes:** ¿etapas ausentes (p. ej., carrito sin checkout)?  
- **Tiempos:** ¿timestamps fuera de orden o en el futuro?



```sql
-- 1) Posibles duplicados exactos por usuario/ts/evento
SELECT user_id, event_name, event_ts, COUNT(*) AS n
FROM events
GROUP BY user_id, event_name, event_ts
HAVING COUNT(*) > 1
ORDER BY n DESC, event_ts;
```

```sql 
-- 2) Eventos "viajeros en el tiempo" (más allá de ahora)
SELECT *
FROM events
WHERE event_ts > NOW();
```

> En SQLite no tenemos disponible la funcion `NOW` en su lugar usamos `datetime('now')`

```sql

-- 3) Usuarios con carrito pero sin checkout/purchase
WITH cart_users AS (
  SELECT DISTINCT user_id FROM events WHERE event_name='add_to_cart'
),
checkout_users AS (
  SELECT DISTINCT user_id FROM events WHERE event_name='begin_checkout'
),
purchase_users AS (
  SELECT DISTINCT user_id FROM events WHERE event_name='purchase'
)
SELECT cu.user_id,
       (cu.user_id IN (SELECT * FROM checkout_users)) AS has_checkout,
       (cu.user_id IN (SELECT * FROM purchase_users)) AS has_purchase
FROM cart_users cu
ORDER BY 1;
```



> Documenta observaciones antes de escribir el funnel.


## Análisis de embudo

----

### Extraer y filtrar eventos relevantes (CTEs)
Las **CTEs** (*Common Table Expressions*, `WITH ... AS (...)`) ayudan a dividir consultas complejas en pasos legibles.


```sql
-- Filtrado por ventana temporal (enero y febrero 2021)
WITH base AS (
  SELECT *
  FROM events
  WHERE event_ts >= '2021-01-01'
    AND event_ts <  '2021-03-01'
), cte_session AS (
  SELECT DISTINCT user_id FROM base WHERE event_name = 'session_start'
), cte_view AS (
  SELECT DISTINCT user_id FROM base WHERE event_name = 'view_item'
), cte_cart AS (
  SELECT DISTINCT user_id FROM base WHERE event_name = 'add_to_cart'
), cte_checkout AS (
  SELECT DISTINCT user_id FROM base WHERE event_name = 'begin_checkout'
), cte_purchase AS (
  SELECT DISTINCT user_id FROM base WHERE event_name = 'purchase'
)
SELECT
  (SELECT COUNT(*) FROM cte_session)  AS session_users,
  (SELECT COUNT(*) FROM cte_view)     AS view_users,
  (SELECT COUNT(*) FROM cte_cart)     AS cart_users,
  (SELECT COUNT(*) FROM cte_checkout) AS checkout_users,
  (SELECT COUNT(*) FROM cte_purchase) AS purchase_users;
```

### Reescribir funnel con “una CTE por etapa” (ejercicio)
**Ejercicio:** Reescribe un funnel (signup → add_to_cart → purchase) usando una CTE por etapa.  
**Pista:** usa `SELECT DISTINCT user_id` en cada CTE y al final cuenta usuarios por etapa.


```sql 
WITH base AS (
  SELECT *
  FROM events
  WHERE event_ts >= '2021-01-01'
    AND event_ts <  '2021-03-01'
), s AS (
  SELECT DISTINCT user_id FROM base WHERE event_name='session_start'
), v AS (
  SELECT DISTINCT user_id FROM base WHERE event_name='view_item'
), c AS (
  SELECT DISTINCT user_id FROM base WHERE event_name='add_to_cart'
), ch AS (
  SELECT DISTINCT user_id FROM base WHERE event_name='begin_checkout'
), p AS (
  SELECT DISTINCT user_id FROM base WHERE event_name='purchase'
), counts AS (
  SELECT
    (SELECT COUNT(*) FROM s)  AS n_session,
    (SELECT COUNT(*) FROM v)  AS n_view,
    (SELECT COUNT(*) FROM c)  AS n_cart,
    (SELECT COUNT(*) FROM ch) AS n_checkout,
    (SELECT COUNT(*) FROM p)  AS n_purchase
)
SELECT
  n_session,
  n_view,
  n_cart,
  n_checkout,
  n_purchase,
  (n_session - n_view)     AS drop_session_to_view,
  (n_view - n_cart)        AS drop_view_to_cart,
  (n_cart - n_checkout)    AS drop_cart_to_checkout,
  (n_checkout - n_purchase)AS drop_checkout_to_purchase,
  ROUND(100.0 * n_purchase / NULLIF(n_session,0), 2) AS cr_session_to_purchase_pct
FROM counts;
```

### Interpretación de resultados
- Identifica el **cuello de botella** (mayor caída).  
- Propón hipótesis (p. ej., fricción en checkout móvil, valor del carrito, medios de pago).  
- Plantea experimento A/B o UX research para esa etapa.


# Análisis de cohortes

----

Definimos **cohorte** por fecha de *signup* (p. ej., mes o semana) y medimos si los usuarios regresan en periodos posteriores (retención D+7, W+1, M+1).  
Conectamos con el funnel calculando métricas por cohort (ej., conversión por cohort mensual).

```sql
-- ==========================================
--  Retención semanal por cohort
-- ==========================================
WITH
-- Cohorte mensual (primer día del mes del signup)
u AS (
  SELECT
    user_id,
    strftime('%Y-%m-01', signup_ts) AS cohort_month
  FROM users
),

-- Normalizamos fechas de eventos
events_norm AS (
  SELECT
    user_id,
    date(event_ts) AS event_date
  FROM events
),

-- Generamos semanas desde 0 hasta 8
grid AS (
  SELECT 0 AS week_offset UNION ALL SELECT 1
  UNION ALL SELECT 2 UNION ALL SELECT 3
  UNION ALL SELECT 4 UNION ALL SELECT 5
  UNION ALL SELECT 6 UNION ALL SELECT 7
  UNION ALL SELECT 8
),

-- Cálculo de retención semanal
retention AS (
  SELECT
    u.cohort_month,
    g.week_offset,
    COUNT(DISTINCT CASE
      WHEN EXISTS (
        SELECT 1
        FROM events_norm en
        WHERE en.user_id = u.user_id
          AND (julianday(en.event_date) - julianday(u.cohort_month)) >= g.week_offset * 7
          AND (julianday(en.event_date) - julianday(u.cohort_month)) < (g.week_offset + 1) * 7
      )
      THEN u.user_id END
    ) AS retained_users,
    COUNT(DISTINCT u.user_id) AS cohort_users
  FROM u
  CROSS JOIN grid g
  GROUP BY u.cohort_month, g.week_offset
)

-- Resultado final con porcentaje de retención
SELECT
  cohort_month,
  week_offset,
  retained_users,
  cohort_users,
  CASE
    WHEN cohort_users = 0 THEN 0
    ELSE ROUND(100.0 * retained_users / cohort_users, 2)
  END AS retention_pct
FROM retention
ORDER BY cohort_month, week_offset


```

**Tabla para mapa de calor**

```sql

 --- Retención semanal por cohort (tabla ancha para heatmap) ---

WITH
-- Cohortes por mes
u AS (
  SELECT
    user_id,
    strftime('%Y-%m-01', signup_ts) AS cohort_month
  FROM users
),

-- Generamos offsets de semanas (0 a 8)
grid AS (
  SELECT 0 AS week_offset UNION ALL SELECT 1
  UNION ALL SELECT 2 UNION ALL SELECT 3
  UNION ALL SELECT 4 UNION ALL SELECT 5
  UNION ALL SELECT 6 UNION ALL SELECT 7
  UNION ALL SELECT 8
),

-- Retención semanal
retention AS (
  SELECT
    u.cohort_month,
    g.week_offset,
    COUNT(DISTINCT CASE
      WHEN EXISTS (
        SELECT 1 FROM events e
        WHERE e.user_id = u.user_id
          AND julianday(e.event_ts) - julianday(u.cohort_month) >= g.week_offset * 7
          AND julianday(e.event_ts) - julianday(u.cohort_month) < (g.week_offset + 1) * 7
      )
      THEN u.user_id END
    ) AS retained_users,
    COUNT(DISTINCT u.user_id) AS cohort_users
  FROM u
  CROSS JOIN grid g
  GROUP BY u.cohort_month, g.week_offset
),

base AS (
  SELECT
    cohort_month,
    week_offset,
    CASE WHEN cohort_users = 0 THEN 0
         ELSE ROUND(100.0 * retained_users / cohort_users, 2)
    END AS retention_pct
  FROM retention
)

-- Pivot para tabla final
SELECT
  cohort_month,
  MAX(CASE WHEN week_offset=0 THEN retention_pct END) AS w0,
  MAX(CASE WHEN week_offset=1 THEN retention_pct END) AS w1,
  MAX(CASE WHEN week_offset=2 THEN retention_pct END) AS w2,
  MAX(CASE WHEN week_offset=3 THEN retention_pct END) AS w3,
  MAX(CASE WHEN week_offset=4 THEN retention_pct END) AS w4,
  MAX(CASE WHEN week_offset=5 THEN retention_pct END) AS w5,
  MAX(CASE WHEN week_offset=6 THEN retention_pct END) AS w6,
  MAX(CASE WHEN week_offset=7 THEN retention_pct END) AS w7,
  MAX(CASE WHEN week_offset=8 THEN retention_pct END) AS w8
FROM base
GROUP BY cohort_month
ORDER BY cohort_month;


```

**Segmentación por plan**

```sql
-- ===============================================
-- Retención semanal segmentada por plan 
-- ===============================================
WITH
-- Cohorte mensual y plan
u AS (
  SELECT
    user_id,
    strftime('%Y-%m-01', signup_ts) AS cohort_month,
    plan
  FROM users
),

-- Generamos semanas 0..8 manualmente (sin generate_series)
grid AS (
  SELECT 0 AS week_offset UNION ALL SELECT 1
  UNION ALL SELECT 2 UNION ALL SELECT 3
  UNION ALL SELECT 4 UNION ALL SELECT 5
  UNION ALL SELECT 6 UNION ALL SELECT 7
  UNION ALL SELECT 8
),

-- Cálculo de retención semanal por cohorte y plan
retention AS (
  SELECT
    u.cohort_month,
    u.plan,
    g.week_offset,
    COUNT(DISTINCT CASE
      WHEN EXISTS (
        SELECT 1
        FROM events e
        WHERE e.user_id = u.user_id
          AND (julianday(e.event_ts) - julianday(u.cohort_month)) >= g.week_offset * 7
          AND (julianday(e.event_ts) - julianday(u.cohort_month)) < (g.week_offset + 1) * 7
      )
      THEN u.user_id END
    ) AS retained_users,
    COUNT(DISTINCT u.user_id) AS cohort_users
  FROM u
  CROSS JOIN grid g
  GROUP BY u.cohort_month, u.plan, g.week_offset
)

-- Resultado final con porcentaje de retención
SELECT
  cohort_month,
  plan,
  week_offset,
  CASE
    WHEN cohort_users = 0 THEN 0
    ELSE ROUND(100.0 * retained_users / cohort_users, 2)
  END AS retention_pct
FROM retention
ORDER BY cohort_month, plan, week_offset;

```

## 🤔💬 Momento de reflexionar en lo aprendido

----

Kahoot time!

## 🚀 Para seguir aprendiendo :

---

- 📚 Vuelve a revisar este notebook y trata resolver por tu cuenta  nuevamente
- 💬 Recuerda que en Discord puedes dejar todos tus comentarios y dudas sobre el contenido del sprint en [`Discord`](https://discord.com/channels/1081207584104656986/1420849538196836472).
    - 📝 Si tienes preguntas sobre tu proyecto, usa el canal [`#project`](https://discord.com/channels/1081207584104656986/1420848813186351134) para recibir ayuda y compartir ideas.
    - 🤝 Aprovecha el espacio de `CoLearning` para aclarar tus dudas junto con otros estudiantes e instructores: [Co-Learning](https://discord.com/channels/1081207584104656986/1197953851391746119).
    - En tus preguntas recuerda etiquetar a `@Dataconsulta` y ubica tu pregunta de acuerdo a `Sprint/Capitulo/Seccion`
- 📅 ¿Necesitas ayuda personalizada? Puedes agendar una sesión `1:1` conmigo aquí: [1:1 Roman Castillo](https://scheduler.zoom.us/roman-castillo/1-1-roman-castillo).

- Por último hazme paro y responde la encuesta al final de la sesión, me sirve para poder ayudarte mejor 

¡Sigue practicando y no dudes en pedir apoyo cuando lo necesites! 💪✨